# ArcGIS API for Python Basics

### It Starts with an Import....

#### Importing the API

- most of the time you just need to import the `GIS` object

In [ ]:
from arcgis.gis import GIS

### Once Loaded It's Time to begin!

## The Way of the ArcGIS API for Python

<img src="./img/waysofscience.jpg" width=700/>

- The Python API allows administrators to manage, update and control what happens on your server
- Script from your favorite IDE or Notebook environment
- Cross platform support

## What Can We Manage?

<table  style='font-family:"Courier New", Courier, monospace; font-size:200%' width=50%>
  
  <tr>
    <td>Users</td>
    <td><img src="./img/users.png", width=50/></td>
  </tr>
  <tr>
    <td>Content</td>
    <td><img src="./img/content.png", width=50/></td>
  </tr>
  <tr>
    <td>Infrastructure</td>
    <td><img src="./img/infrastructure.png", width=50/></td>
  </tr>
  <tr>
    <td>Groups</td>
    <td><img src="./img/groups.png" width=50/></td>
  </tr>
 
</table>

## Let's Understand How we Can Access the System

#### Supported Authentication:

1. **Anonymous Connection** - no credentials are provided

    **Connects to ArcGIS Online**
    ```python
    from arcgis.gis import GIS
    gis = GIS()
    ```
    **Connects to Enterprise**

    ```python
    from arcgis.gis import GIS
    gis = GIS(url="https://mynotrealsite.com/wa/portal")
    ```

2. **Built-In** - provides a Username/Password
   
    **Connects to ArcGIS Online**
    ```python
    from arcgis.gis import GIS
    gis = GIS(username="AFakeAccount", password="SuperSecret")
    ```

    **Connects to Enterprise**
    ```python
    from arcgis.gis import GIS
    gis = GIS(url="https://mynotrealsite.com/wa/portal", username="AFakeAccount", password="SuperSecret")
    ```
    
3. **Web-tier authentication with LDAP**

    **Connecting to Enterprise**
    ```python
    ldap = GIS("https://portalname.domain.com/webadapter_name", "amy", "password")
    ```

5. **Portal-tier authentication with Active Directory**

    - When connecting using an enterprise user account from an Active Directory, specify your username as domain\\username or username@domain

    **Enterprise Active Directory Example**
    ```python
    gis = GIS("https://portalname.domain.com/webadapter_name", "DOMAIN\\Publisher", "password")
    ```

7. **Portal-tier authentication with LDAP**

    ```python
    gis = GIS("https://portalname.domain.com/webadapter_name", "sharing1", "password")
    ```

8. Web-tier authentication secured with PKI

    **Key/Cert File**
    ```python
    gis = GIS("https://portalname.domain.com/webcontext", 
          key_file="C:\\path\\to\\key.pem",
          cert_file="C:\\path\\to\\cert.pem")
    ```
    **Via PFX File**
    ```python
    gis = GIS("https://portalname.domain.com/webcontext", 
          cert_file="C:\\path\\to\\mycert.pfx",
          password="secret.password")
    ```


## Protecting Credentials

### Profiles Protect Credentials
- using `profiles` will help protect username and passwords.  
- prevents accidental sharing
- Leverages the Operating System's credential store
- Available on Mac, Windows and Linux
- Helps you share workflows but not passwords!

1. Create a `GIS` object with the extra `profile` parameter

```python
gis = GIS(url="https://www.mysite.com/portal", 
          username='fakeaccount', 
          password='fakepassword', 
          profile='portal_profile')
```

2. Now connect using the `profile`

```python
gis = GIS(profile='portal_profile')
```

**What Happened?**

Instead of keeping your password in plain text, now we leverage the operating system's credential store for the logged in user.  The credentials never get passed on when you use profiles.

Moving forward I will always use profiles

## API Key/Developer Credentials

In [ ]:
from arcgis.gis.admin import AGOLAdminManager
from arcgis.gis.admin._stokenmgr import  TokenPrivilege
import datetime as _dt

In [ ]:
gis = GIS(profile='your_online_profile')

In [ ]:
admin:AGOLAdminManager = gis.admin
dev_creds = admin.developer_credentials

In [ ]:
api_credential = dev_creds.create(title='my first api key',
                                   privileges=[TokenPrivilege.PORTAL_ADMIN_VIEWITEMS, 
                                               TokenPrivilege.PORTAL_USER_CREATEITEM],
                                   referers=['http'],
                                   expiration=_dt.datetime.now() + _dt.timedelta(weeks=20))
api_credential

In [ ]:
token = api_credential.generate_token(slot=1)
token

In [ ]:
GIS(token=token['access_token']).users.me

In [ ]:
api_credential.delete()

## Working with SSL Certificates

### How SSL Certificates Work in the API

- Leverages `truststore` to obtain your operating systems certificate store
    + Windows Cert Store or Mac keychain for example
- This means whatever your PC trusts so shall the API
- When you update your machines certificates, the Python API will pick it up automatically

#### Using a CA_BUNDLE specifying SSL certificates

- There are times where you need to specify your own SSL certificate bundle

```python
certs = r"./CA_CERTS/cacert.pem"
gis = GIS(profile="your_enterprise_admin_profile", ca_bundles=certs)
```

#### Ignoring SSL certificates

- There are times when you need to ignore SSL verification all together

```python
gis = GIS(profile="your_enterprise_admin_profile", verify_cert=False)
```

- This should not be used in production code!

## Working with the `GIS` Object

The `GIS` object allows for administrators to start managing their orgnaization

In [ ]:
from arcgis.gis import GIS
gis = GIS(profile='your_online_profile', verify_cert=False, trust_env=True)

### Important Properties

- `admin` - contains all the administrative functionality
- `users` - provides access to work with the User operations
- `content` - allows for management of Item/Content on our organization
- `groups` - allows for management of the organization's groups
- `servers` - provides registered servers to the Enterprise site/org
    

### Gaining Insights from Queries

<center><img src="./img/content-meme-2026.jpg"></center>

- You do not always use the administrative functions to find insights in our ArcGIS Enterprises or Organizations
- The `advanced_search` functions allow for the gathering of statistics and other information that can give you an insight not easily seen

#### Content Advanced Search

- This search method allows for a fully customizable search experience
- As a user you have full control
- **Don't Let the Name Fool You!**

In [ ]:
cm = gis.content

##### Searching

simple searches

In [ ]:
items = cm.advanced_search('title: battle AND  (type:"feature service")', max_items=-1, 
                           sort_field='avgRating', sort_order='desc')['results']
items[10:20]

##### Searching for a Specific User's Content

- Find all the feature services owned by a given user

In [ ]:
items = cm.advanced_search('(type:"feature service") AND owner:andrew57', max_items=-1, 
                           sort_field='avgRating', sort_order='desc')['results']
items[1:10]

##### Find All the New Items Created within the Last 5 Days

- Finds the count of all the items created within my organization within the last 5 days.

In [ ]:
import datetime as _dt

now =_dt.datetime.now(_dt.timezone.utc)
then = now - _dt.timedelta(days=5)

In [ ]:
cm.advanced_search(
    f"created: [{int(then.timestamp()* 1000)} TO {int(now.timestamp()* 1000)}] AND accountid:{gis.properties.id}", 
    return_count=True)

In [ ]:
search_result = cm.advanced_search(
    f"created: [{int(then.timestamp()* 1000)} TO {int(now.timestamp()* 1000)}] AND accountid:{gis.properties.id}", 
                   sort_order = "asc", count_fields = "type")
search_result 

##### Visualize it Better with Matplotlib

In [ ]:
import matplotlib.pyplot as plt
# Get the Data
data = search_result.get("counts")[0].get("fieldValues")
labels = [item['value'] for item in data]
counts = [item['count'] for item in data]
# Reverse the order to get the highest first
labels.reverse()
counts.reverse()
# Create the Bar Chart
plt.barh(labels, counts, color='skyblue')
# Add labels and title
plt.xlabel('Count')
plt.ylabel('Type')
plt.title('New Items by Type')
# Adjust layout to ensure labels are not truncated
plt.tight_layout()